# Donut v2 eval — Drive edition (apples-to-apples vs Claude)

Inference-only notebook for the fine-tuned Donut v2 (200M params, XML target
format). Converts XML output → JSON in-notebook so the saved JSONL is
schema-compatible with Qwen/InternVL3 → same `make eval-replay` script
scores all three.

## One-time Drive setup

```
MyDrive/ttb_sft/
├── donut_v2/
│   └── model/                  ← from sft_donut_v2.ipynb training run
└── eval/
    └── cola_sample.csv         ← already there
```

## How to run

1. Runtime → **T4 or A100** (Donut is small, even T4 is fast)
2. Disconnect and delete runtime
3. Runtime → **Run all** (1st click — install, kernel auto-crashes)
4. Runtime → **Run all** (2nd click — Drive permission dialog → accept; runs end-to-end)
5. `donut_outputs.jsonl` auto-downloads. Then locally:
   ```
   make eval-replay-donut
   ```

## Expected speed

Donut at 200M params should be **~10-15× faster than Qwen 7B**. Target
latency: ~0.3-0.5s/image, total ~30 sec for 90 images.

## 1. Install dependencies

In [ ]:
# Donut needs standard transformers + accelerate. No peft (full fine-tune,
# not LoRA). No bnb. Minimal stack.
import importlib.util, subprocess, sys, os


def _have(mod): return importlib.util.find_spec(mod) is not None
def _pinned(mod, want):
    try: return __import__(mod).__version__ == want
    except Exception: return False
def _numpy_ok():
    try:
        import numpy
        p = numpy.__version__.split(".")
        return int(p[0]) == 2 and int(p[1]) < 2
    except Exception: return False


_required = ["accelerate", "sentencepiece", "PIL"]
_ready = (_numpy_ok() and _pinned("transformers", "4.51.3")
          and all(_have(m) for m in _required))

if _ready:
    import numpy, transformers
    print(f"✓ Set up — numpy {numpy.__version__}, transformers {transformers.__version__}")
else:
    print("⏳ Installing dependencies (~2 min). Kernel will auto-restart at end.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
    subprocess.check_call([sys.executable, "-m", "pip", "install",
        "transformers==4.51.3", "accelerate>=0.30",
        "protobuf>=5.27", "sentencepiece", "pillow", "tqdm"])
    subprocess.check_call([sys.executable, "-m", "pip", "install",
        "--force-reinstall", "--no-deps", "numpy>=2.0,<2.2"])
    print()
    print("=" * 70)
    print("✅ Install complete. Auto-restarting kernel.")
    print("   Wait for 'session crashed' banner, then Runtime → Run all AGAIN.")
    print("=" * 70)
    os.kill(os.getpid(), 9)

## 2. Mount Drive + verify weights and CSV are in place

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT  = Path("/content/drive/MyDrive/ttb_sft")
MODEL_DIR   = DRIVE_ROOT / "donut_v2" / "model"
CSV_PATH    = DRIVE_ROOT / "eval" / "cola_sample.csv"
OUTPUT_DIR  = DRIVE_ROOT / "eval"

# Donut is a full fine-tune (no LoRA) — we look for a saved
# VisionEncoderDecoder model directory.
required_files = ["config.json", "tokenizer_config.json"]
missing = [f for f in required_files if not (MODEL_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"\nMissing Donut model files in {MODEL_DIR}:\n  {missing}\n"
        f"Run sft_donut_v2.ipynb first to produce it."
    )
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"\nMissing CSV at {CSV_PATH}\n"
        f"Upload test/eval/data/cola_sample.csv to MyDrive/ttb_sft/eval/."
    )

print(f"✓ Model:  {MODEL_DIR}")
print(f"✓ CSV:    {CSV_PATH} ({CSV_PATH.stat().st_size:,} bytes)")
print(f"✓ Output: {OUTPUT_DIR}")

## 3. Load Donut + processor

In [ ]:
import torch
from transformers import VisionEncoderDecoderModel, DonutProcessor

print(f"Loading Donut v2 from {MODEL_DIR}...")
processor = DonutProcessor.from_pretrained(str(MODEL_DIR))
model = VisionEncoderDecoderModel.from_pretrained(
    str(MODEL_DIR),
    torch_dtype=torch.bfloat16,
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"✓ Loaded on {device}: {n_params/1e6:.0f}M params")

## 4. Load CSV + download test images

In [ ]:
import csv, urllib.request, concurrent.futures

rows = list(csv.DictReader(open(CSV_PATH)))
print(f"Loaded {len(rows)} rows")

IMG_DIR = Path("/content/donut_eval_images")
IMG_DIR.mkdir(exist_ok=True)
CDN = "https://dyuie4zgfxmt6.cloudfront.net/{}.webp"

def _dl(row):
    image_id = row["ttb_image_id"]
    dest = IMG_DIR / f"{image_id}.webp"
    if dest.exists() and dest.stat().st_size > 0:
        return row, True
    try:
        req = urllib.request.Request(CDN.format(image_id),
                                     headers={"User-Agent": "ttb-eval/0.1"})
        with urllib.request.urlopen(req, timeout=20) as r:
            dest.write_bytes(r.read())
        return row, True
    except Exception as e:
        print(f"  download fail {image_id}: {e}")
        return row, False

downloaded = []
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as ex:
    for row, ok in ex.map(_dl, rows):
        if ok:
            row["_image_path"] = str(IMG_DIR / f"{row['ttb_image_id']}.webp")
            downloaded.append(row)
print(f"✓ Downloaded {len(downloaded)} / {len(rows)} images")

## 5. Run Donut extraction (XML output → JSON conversion)

Donut v2 emits XML-tagged output (matches its training-time target format).
We convert that to the same JSON schema Qwen/InternVL3 produce so the
downstream `make eval-replay-donut` sees the same shape and computes the
same metrics.

In [ ]:
import json, re, time, statistics
from PIL import Image


def _xml_to_json(xml: str):
    """Parse Donut's XML output into the JSON schema used by the rules engine."""
    if not xml or "<s_ttb>" not in xml:
        return None
    field_map = {
        "s_brand":   "Brand name",
        "s_class":   "Class & type",
        "s_abv":     "Alcohol content",
        "s_net":     "Net contents",
        "s_bottler": "Bottler name/address",
        "s_origin":  "Country of origin",
    }
    fields = {}
    for tag, name in field_map.items():
        m = re.search(rf"<{tag}>(.*?)</{tag}>", xml, re.DOTALL)
        if m and m.group(1).strip():
            fields[name] = {"value": m.group(1).strip(), "confidence": 0.9}

    warning_present = "<s_warning_present>true</s_warning_present>" in xml
    warning_text_m  = re.search(r"<s_warning>(.*?)</s_warning>", xml, re.DOTALL)
    casing_caps     = "<s_casing_caps>true</s_casing_caps>" in xml
    body_bold       = "<s_body_bold>true</s_body_bold>" in xml
    return {
        "fields": fields,
        "government_warning": {
            "present":            warning_present,
            "detected_text":      warning_text_m.group(1).strip() if warning_text_m else "",
            "casing_all_caps":    casing_caps,
            "heading_bold":       warning_present,
            "body_bold":          body_bold,
            "approx_font_mm":     None,
            "contrast_ok":        True,
            "separate_and_apart": True,
        },
        "image_quality": {"score": 0.85, "legible": True, "note": None},
    }


@torch.no_grad()
def _donut_extract(image_path, beverage_type):
    img = Image.open(image_path).convert("RGB")
    pixel_values = processor(images=[img], return_tensors="pt").pixel_values.to(torch.bfloat16).to(model.device)
    out = model.generate(
        pixel_values,
        decoder_start_token_id=model.config.decoder_start_token_id,
        max_length=1024,
        do_sample=False,
        early_stopping=True,
        num_beams=1,
        eos_token_id=processor.tokenizer.eos_token_id,
        pad_token_id=processor.tokenizer.pad_token_id,
    )
    text = processor.tokenizer.decode(out[0], skip_special_tokens=False)
    return _xml_to_json(text), text


print("Sanity check on first image...")
t0 = time.time()
pred0, _ = _donut_extract(downloaded[0]["_image_path"],
                          (downloaded[0].get("beverage_type") or "wine").lower())
print(f"  ✓ First image: {time.time()-t0:.2f}s")

outputs_jsonl = []
n_parsed = n_unparseable = 0
latencies = []

print(f"\nExtracting on {len(downloaded)} images...")
for i, row in enumerate(downloaded, 1):
    bev = (row.get("beverage_type") or "wine").lower()
    t0 = time.time()
    pred, raw_text = _donut_extract(row["_image_path"], bev)
    dt = time.time() - t0
    latencies.append(dt)
    outputs_jsonl.append({
        "image_filename": row.get("image_filename") or (row["ttb_image_id"] + ".webp"),
        "ttb_image_id":   row["ttb_image_id"],
        "beverage_type":  bev,
        "raw_output":     raw_text,
        "parsed_output":  pred,
        "latency_sec":    round(dt, 3),
    })
    if pred is None: n_unparseable += 1
    else:            n_parsed += 1
    if i % 10 == 0:
        print(f"  [{i}/{len(downloaded)}] parsed={n_parsed} unparseable={n_unparseable} "
              f"latency_mean={statistics.mean(latencies):.2f}s")

print()
print("=" * 70)
print(f"Done. Parsed: {n_parsed} / Unparseable: {n_unparseable}")
print(f"Latency mean / p95: {statistics.mean(latencies):.2f}s / "
      f"{sorted(latencies)[int(len(latencies)*0.95)]:.2f}s")
print("=" * 70)

## 6. Save outputs to Drive AND auto-download

In [ ]:
jsonl_path = OUTPUT_DIR / "donut_outputs.jsonl"
with open(jsonl_path, "w") as f:
    for rec in outputs_jsonl:
        f.write(json.dumps(rec) + "\n")
print(f"✓ Saved {jsonl_path} ({len(outputs_jsonl)} rows)")

from google.colab import files
files.download(str(jsonl_path))

print()
print("Next on your Mac:")
print("  make eval-replay-donut")
print("    → writes test/eval/donut_report.json + side-by-side vs Claude.")